# Day 2: Advanced Multi-Agent Patterns & Root Cause Analysis

## Advanced Multi-Agent AI Systems Training
### For Support & Meta Engineers

---

## 🎯 Learning Objectives

By the end of this session, you will:
1. Implement hierarchical multi-agent systems
2. Build evidence-based reasoning pipelines
3. Create a 5-agent root cause analysis system
4. Apply production guardrails and safety mechanisms
5. Deploy production-ready agent systems

## 📋 Session Outline

1. **Recap & Advanced Patterns** (20 min)
2. **Hierarchical Agent Systems** (30 min)
3. **Evidence-Based Reasoning** (30 min)
4. **Building the RCA System** (60 min)
5. **Production Guardrails** (30 min)
6. **Hands-on Exercises** (50 min)

---

## Part 1: Setup & Recap

### 1.1 Environment Setup

In [ ]:
import sys
import os
import json
from pathlib import Path
from datetime import datetime

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"✓ Project root: {project_root}")
print(f"✓ Python version: {sys.version.split()[0]}")
print(f"✓ Day 2: Advanced Patterns & RCA")

### 1.2 Initialize LLM

In [ ]:
from src.llm import get_llm, print_llm_stats
from src.agents import Agent, AgentRole, Orchestrator, WorkflowStep
from src.utils import LogParser, MetricsCollector

llm = get_llm(force_mock=False, deterministic=True, verbose=True)

print("\n✓ LLM initialized and ready")

### 1.3 Load Sample Data

In [ ]:
with open('data/sample_incidents.json', 'r') as f:
    incidents = json.load(f)

with open('data/sample_logs.txt', 'r') as f:
    sample_logs = f.read()

print(f"✓ Loaded {len(incidents)} incidents")
print(f"✓ Loaded {len(sample_logs.splitlines())} log lines")

---

## Part 2: Advanced Agent Patterns

### 2.1 Hierarchical Agent Systems

**Pattern**: Coordinator agent manages multiple worker agents

```
         Coordinator
              |
    +---------+---------+
    |         |         |
 Worker1   Worker2   Worker3
```

**Benefits**:
- Parallel execution of independent tasks
- Centralized decision-making
- Dynamic task allocation
- Better resource management

In [ ]:
coordinator = Agent(
    name="RCACoordinator",
    role=AgentRole.COORDINATOR,
    llm=llm,
    system_prompt="""You are a root cause analysis coordinator. 
    Your job is to:
    1. Break down complex incidents into sub-tasks
    2. Assign tasks to specialized worker agents
    3. Synthesize results into coherent RCA
    4. Identify gaps and request additional analysis
    """
)

print(f"✓ Created coordinator: {coordinator}")
print(f"  Role: {coordinator.role.value}")

### 2.2 Evidence-Based Reasoning

**Key Principle**: All conclusions must be backed by evidence

**Evidence Types**:
1. **Log entries**: Timestamped error messages
2. **Metrics**: CPU, memory, latency data
3. **Events**: Deployments, config changes
4. **Patterns**: Recurring issues, correlations

**Reasoning Chain**:
```
Evidence → Pattern → Hypothesis → Validation → Conclusion
```

In [ ]:
class EvidenceChain:
    def __init__(self):
        self.evidence = []
        self.hypotheses = []
        self.validations = []
    
    def add_evidence(self, evidence_type: str, data: dict, confidence: float):
        self.evidence.append({
            'type': evidence_type,
            'data': data,
            'confidence': confidence,
            'timestamp': datetime.now().isoformat()
        })
    
    def add_hypothesis(self, hypothesis: str, supporting_evidence: list):
        self.hypotheses.append({
            'hypothesis': hypothesis,
            'evidence_ids': supporting_evidence,
            'timestamp': datetime.now().isoformat()
        })
    
    def validate(self, hypothesis_id: int, is_valid: bool, reasoning: str):
        self.validations.append({
            'hypothesis_id': hypothesis_id,
            'is_valid': is_valid,
            'reasoning': reasoning,
            'timestamp': datetime.now().isoformat()
        })
    
    def get_summary(self):
        return {
            'total_evidence': len(self.evidence),
            'total_hypotheses': len(self.hypotheses),
            'validated': len(self.validations),
            'evidence': self.evidence,
            'hypotheses': self.hypotheses,
            'validations': self.validations
        }

evidence_chain = EvidenceChain()
evidence_chain.add_evidence(
    'log_error',
    {'message': 'Database connection timeout', 'count': 50},
    confidence=0.95
)
evidence_chain.add_hypothesis(
    'Connection pool exhaustion due to increased load',
    supporting_evidence=[0]
)

print("Evidence Chain Example:")
print(json.dumps(evidence_chain.get_summary(), indent=2))

---

## Part 3: Building the RCA System

### 3.1 System Architecture

Our RCA system has **5 specialized agents**:

1. **Log Parser Agent**: Extracts structured data from logs
2. **Pattern Detector Agent**: Identifies anomalies and patterns
3. **Correlation Agent**: Links related events
4. **Hypothesis Agent**: Generates root cause hypotheses
5. **Validator Agent**: Validates hypotheses against evidence

```
Logs → Parser → Pattern Detector → Correlator → Hypothesis Generator → Validator → RCA Report
```

### 3.2 Parse Logs with Utility

In [ ]:
log_parser_util = LogParser()
parsed_entries = log_parser_util.parse_logs(sample_logs)

print(f"✓ Parsed {len(parsed_entries)} log entries\n")

analysis = log_parser_util.analyze_logs(parsed_entries)
print("Log Analysis:")
print(json.dumps(analysis, indent=2))

print("\n" + log_parser_util.get_error_summary(parsed_entries))

### 3.3 Create RCA Agents

In [ ]:
log_parser_agent = Agent(
    name="LogParser",
    role=AgentRole.LOG_PARSER,
    llm=llm
)

pattern_detector = Agent(
    name="PatternDetector",
    role=AgentRole.PATTERN_DETECTOR,
    llm=llm
)

correlator = Agent(
    name="EventCorrelator",
    role=AgentRole.CORRELATOR,
    llm=llm
)

hypothesis_generator = Agent(
    name="HypothesisGenerator",
    role=AgentRole.HYPOTHESIS_GENERATOR,
    llm=llm
)

validator = Agent(
    name="HypothesisValidator",
    role=AgentRole.VALIDATOR,
    llm=llm
)

print("✓ Created 5 RCA agents:")
for agent in [log_parser_agent, pattern_detector, correlator, hypothesis_generator, validator]:
    print(f"  - {agent.name} ({agent.role.value})")

### 3.4 Build RCA Workflow

In [ ]:
rca_orchestrator = Orchestrator(name="RootCauseAnalysisWorkflow")

rca_orchestrator.register_agent("log_parser", log_parser_agent)
rca_orchestrator.register_agent("pattern_detector", pattern_detector)
rca_orchestrator.register_agent("correlator", correlator)
rca_orchestrator.register_agent("hypothesis_generator", hypothesis_generator)
rca_orchestrator.register_agent("validator", validator)

rca_workflow = [
    WorkflowStep(
        name="parse_logs",
        agent_name="log_parser"
    ),
    WorkflowStep(
        name="detect_patterns",
        agent_name="pattern_detector",
        depends_on=["parse_logs"]
    ),
    WorkflowStep(
        name="correlate_events",
        agent_name="correlator",
        depends_on=["parse_logs", "detect_patterns"]
    ),
    WorkflowStep(
        name="generate_hypothesis",
        agent_name="hypothesis_generator",
        depends_on=["parse_logs", "detect_patterns", "correlate_events"]
    ),
    WorkflowStep(
        name="validate_hypothesis",
        agent_name="validator",
        depends_on=["generate_hypothesis"]
    )
]

rca_orchestrator.build_workflow(rca_workflow)

print(rca_orchestrator.visualize_workflow())

### 3.5 Execute RCA Workflow

In [ ]:
rca_input = {
    'incident': incidents[0],
    'logs': sample_logs,
    'log_analysis': analysis
}

print("\n" + "="*60)
print(f"Starting RCA for: {incidents[0]['title']}")
print("="*60 + "\n")

rca_result = rca_orchestrator.execute_workflow(rca_input, verbose=True)

print("\n" + "="*60)
print("RCA Workflow Complete")
print("="*60)
print(f"Status: {rca_result['status']}")
print(f"Duration: {rca_result['duration_seconds']:.2f}s")
print(f"Steps Executed: {rca_result['steps_executed']}")

### 3.6 Examine RCA Results

In [ ]:
print("\n📊 LOG PARSING RESULT:")
print("="*60)
parse_result = rca_orchestrator.get_step_result("parse_logs")
print(parse_result['response'])

print("\n📊 PATTERN DETECTION RESULT:")
print("="*60)
pattern_result = rca_orchestrator.get_step_result("detect_patterns")
print(pattern_result['response'])

print("\n📊 CORRELATION RESULT:")
print("="*60)
corr_result = rca_orchestrator.get_step_result("correlate_events")
print(corr_result['response'])

print("\n📊 HYPOTHESIS RESULT:")
print("="*60)
hyp_result = rca_orchestrator.get_step_result("generate_hypothesis")
print(hyp_result['response'])

print("\n📊 VALIDATION RESULT:")
print("="*60)
val_result = rca_orchestrator.get_step_result("validate_hypothesis")
print(val_result['response'])

### 3.7 Generate RCA Report

In [ ]:
def generate_rca_report(orchestrator, incident):
    report = []
    report.append("="*70)
    report.append("ROOT CAUSE ANALYSIS REPORT")
    report.append("="*70)
    report.append(f"")
    report.append(f"Incident ID: {incident['id']}")
    report.append(f"Title: {incident['title']}")
    report.append(f"Severity: {incident['severity']}")
    report.append(f"Affected Users: {incident['affected_users']}")
    report.append(f"Timestamp: {incident['timestamp']}")
    report.append(f"")
    report.append("-"*70)
    report.append("ANALYSIS SUMMARY")
    report.append("-"*70)
    report.append(f"")
    
    hyp_result = orchestrator.get_step_result("generate_hypothesis")
    if hyp_result:
        report.append("ROOT CAUSE HYPOTHESIS:")
        report.append(hyp_result['response'])
        report.append(f"")
    
    val_result = orchestrator.get_step_result("validate_hypothesis")
    if val_result:
        report.append("VALIDATION:")
        report.append(val_result['response'])
        report.append(f"")
    
    pattern_result = orchestrator.get_step_result("detect_patterns")
    if pattern_result:
        report.append("KEY PATTERNS DETECTED:")
        report.append(pattern_result['response'])
        report.append(f"")
    
    report.append("-"*70)
    report.append("RECOMMENDATIONS")
    report.append("-"*70)
    report.append("1. Immediate: Address identified root cause")
    report.append("2. Short-term: Implement monitoring for similar patterns")
    report.append("3. Long-term: Review system architecture for prevention")
    report.append(f"")
    report.append("="*70)
    report.append(f"Report Generated: {datetime.now().isoformat()}")
    report.append("="*70)
    
    return "\n".join(report)

rca_report = generate_rca_report(rca_orchestrator, incidents[0])
print(rca_report)

---

## Part 4: Production Guardrails

### 4.1 Confidence Scoring

Always track confidence levels for agent outputs:

In [ ]:
import re

def extract_confidence(response_text):
    patterns = [
        r'confidence[:\s]+(\d+)%',
        r'(\d+)%\s+confidence',
        r'confidence[:\s]+(\d+\.\d+)'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, response_text, re.IGNORECASE)
        if match:
            confidence = float(match.group(1))
            if confidence > 1:
                confidence = confidence / 100
            return confidence
    
    return 0.5

def should_require_human_review(confidence, severity):
    if confidence < 0.7:
        return True, "Low confidence - requires human review"
    
    if severity in ['P0', 'P1'] and confidence < 0.9:
        return True, "Critical incident - requires human validation"
    
    return False, "Confidence sufficient for automated action"

hyp_result = rca_orchestrator.get_step_result("generate_hypothesis")
confidence = extract_confidence(hyp_result['response'])
needs_review, reason = should_require_human_review(confidence, incidents[0]['severity'])

print(f"Hypothesis Confidence: {confidence:.2%}")
print(f"Requires Human Review: {needs_review}")
print(f"Reason: {reason}")

### 4.2 Evidence Validation

Validate that hypotheses are backed by evidence:

In [ ]:
def validate_evidence_chain(orchestrator):
    validation_results = {
        'has_logs': False,
        'has_patterns': False,
        'has_correlation': False,
        'has_hypothesis': False,
        'has_validation': False,
        'is_complete': False
    }
    
    if orchestrator.get_step_result("parse_logs"):
        validation_results['has_logs'] = True
    
    if orchestrator.get_step_result("detect_patterns"):
        validation_results['has_patterns'] = True
    
    if orchestrator.get_step_result("correlate_events"):
        validation_results['has_correlation'] = True
    
    if orchestrator.get_step_result("generate_hypothesis"):
        validation_results['has_hypothesis'] = True
    
    if orchestrator.get_step_result("validate_hypothesis"):
        validation_results['has_validation'] = True
    
    validation_results['is_complete'] = all([
        validation_results['has_logs'],
        validation_results['has_patterns'],
        validation_results['has_hypothesis'],
        validation_results['has_validation']
    ])
    
    return validation_results

evidence_validation = validate_evidence_chain(rca_orchestrator)
print("Evidence Chain Validation:")
print(json.dumps(evidence_validation, indent=2))

if evidence_validation['is_complete']:
    print("\n✅ Evidence chain is complete and valid")
else:
    print("\n⚠️  Evidence chain is incomplete - additional analysis required")

### 4.3 Rate Limiting & Circuit Breakers

In [ ]:
from collections import deque
from datetime import datetime, timedelta

class RateLimiter:
    def __init__(self, max_calls, time_window_seconds):
        self.max_calls = max_calls
        self.time_window = timedelta(seconds=time_window_seconds)
        self.calls = deque()
    
    def is_allowed(self):
        now = datetime.now()
        
        while self.calls and now - self.calls[0] > self.time_window:
            self.calls.popleft()
        
        if len(self.calls) < self.max_calls:
            self.calls.append(now)
            return True
        
        return False
    
    def get_stats(self):
        return {
            'current_calls': len(self.calls),
            'max_calls': self.max_calls,
            'time_window_seconds': self.time_window.total_seconds()
        }

class CircuitBreaker:
    def __init__(self, failure_threshold=5, timeout_seconds=60):
        self.failure_threshold = failure_threshold
        self.timeout = timedelta(seconds=timeout_seconds)
        self.failures = 0
        self.last_failure_time = None
        self.state = 'CLOSED'
    
    def call(self, func, *args, **kwargs):
        if self.state == 'OPEN':
            if datetime.now() - self.last_failure_time > self.timeout:
                self.state = 'HALF_OPEN'
            else:
                raise Exception(f"Circuit breaker OPEN - too many failures")
        
        try:
            result = func(*args, **kwargs)
            self.on_success()
            return result
        except Exception as e:
            self.on_failure()
            raise e
    
    def on_success(self):
        self.failures = 0
        self.state = 'CLOSED'
    
    def on_failure(self):
        self.failures += 1
        self.last_failure_time = datetime.now()
        
        if self.failures >= self.failure_threshold:
            self.state = 'OPEN'
    
    def get_stats(self):
        return {
            'state': self.state,
            'failures': self.failures,
            'threshold': self.failure_threshold
        }

rate_limiter = RateLimiter(max_calls=10, time_window_seconds=60)
circuit_breaker = CircuitBreaker(failure_threshold=5, timeout_seconds=60)

print("Guardrails Initialized:")
print(f"  Rate Limiter: {rate_limiter.get_stats()}")
print(f"  Circuit Breaker: {circuit_breaker.get_stats()}")

for i in range(3):
    if rate_limiter.is_allowed():
        print(f"  ✓ Call {i+1} allowed")
    else:
        print(f"  ✗ Call {i+1} rate limited")

### 4.4 Audit Trail

In [ ]:
class AuditLogger:
    def __init__(self):
        self.audit_trail = []
    
    def log_decision(self, agent_name, decision, reasoning, confidence):
        self.audit_trail.append({
            'timestamp': datetime.now().isoformat(),
            'agent': agent_name,
            'decision': decision,
            'reasoning': reasoning,
            'confidence': confidence
        })
    
    def log_action(self, action_type, details, approved_by=None):
        self.audit_trail.append({
            'timestamp': datetime.now().isoformat(),
            'type': 'action',
            'action_type': action_type,
            'details': details,
            'approved_by': approved_by
        })
    
    def get_trail(self):
        return self.audit_trail
    
    def export_trail(self, filepath):
        with open(filepath, 'w') as f:
            json.dump(self.audit_trail, f, indent=2)

audit_logger = AuditLogger()

audit_logger.log_decision(
    agent_name="HypothesisGenerator",
    decision="Database connection pool exhaustion",
    reasoning="50+ connection timeout errors in logs",
    confidence=0.85
)

audit_logger.log_action(
    action_type="rollback_deployment",
    details={"from_version": "2.5.1", "to_version": "2.5.0"},
    approved_by="human_operator"
)

print("Audit Trail:")
print(json.dumps(audit_logger.get_trail(), indent=2))

---

## Part 5: Advanced Exercises

### Exercise 1: Add Metric Analysis Agent

**Task**: Create a new agent that analyzes metrics (CPU, memory, latency)

**Requirements**:
- Integrate into RCA workflow after pattern detection
- Identify metric anomalies
- Correlate with log patterns

**Hint**: Create synthetic metric data for testing

In [ ]:
import random

def generate_synthetic_metrics(num_points=50):
    metrics = []
    base_cpu = 30
    base_memory = 60
    base_latency = 200
    
    for i in range(num_points):
        if i > 30:
            cpu = base_cpu + random.randint(40, 60)
            memory = base_memory + random.randint(20, 35)
            latency = base_latency + random.randint(2000, 4000)
        else:
            cpu = base_cpu + random.randint(-5, 10)
            memory = base_memory + random.randint(-5, 10)
            latency = base_latency + random.randint(-50, 100)
        
        metrics.append({
            'timestamp': f"2024-01-27T10:{i:02d}:00",
            'cpu_percent': cpu,
            'memory_percent': memory,
            'latency_ms': latency
        })
    
    return metrics

synthetic_metrics = generate_synthetic_metrics()

metric_analyzer = Agent(
    name="MetricAnalyzer",
    role=AgentRole.PATTERN_DETECTOR,
    llm=llm,
    system_prompt="""You are an expert at analyzing system metrics.
    Identify anomalies in CPU, memory, and latency data.
    Look for: spikes, gradual increases, sudden drops, correlations.
    """
)

rca_orchestrator_ex1 = Orchestrator(name="RCAWithMetrics")
rca_orchestrator_ex1.register_agent("log_parser", log_parser_agent)
rca_orchestrator_ex1.register_agent("pattern_detector", pattern_detector)
rca_orchestrator_ex1.register_agent("metric_analyzer", metric_analyzer)
rca_orchestrator_ex1.register_agent("correlator", correlator)
rca_orchestrator_ex1.register_agent("hypothesis_generator", hypothesis_generator)
rca_orchestrator_ex1.register_agent("validator", validator)

workflow_ex1 = [
    WorkflowStep(name="parse_logs", agent_name="log_parser"),
    WorkflowStep(name="detect_patterns", agent_name="pattern_detector", depends_on=["parse_logs"]),
    WorkflowStep(name="analyze_metrics", agent_name="metric_analyzer", depends_on=["parse_logs"]),
    WorkflowStep(name="correlate_events", agent_name="correlator", depends_on=["detect_patterns", "analyze_metrics"]),
    WorkflowStep(name="generate_hypothesis", agent_name="hypothesis_generator", depends_on=["correlate_events"]),
    WorkflowStep(name="validate_hypothesis", agent_name="validator", depends_on=["generate_hypothesis"])
]

rca_orchestrator_ex1.build_workflow(workflow_ex1)

print(rca_orchestrator_ex1.visualize_workflow())

rca_input_ex1 = {
    'incident': incidents[0],
    'logs': sample_logs,
    'metrics': synthetic_metrics
}

result_ex1 = rca_orchestrator_ex1.execute_workflow(rca_input_ex1, verbose=True)

print("\n📊 METRIC ANALYSIS:")
print("="*60)
metric_result = rca_orchestrator_ex1.get_step_result("analyze_metrics")
print(metric_result['response'])

### Exercise 2: Parallel Agent Execution

**Task**: Modify workflow to run pattern detection and metric analysis in parallel

**Requirements**:
- Both agents should run simultaneously after log parsing
- Correlator waits for both to complete
- Measure time savings

**Hint**: Remove dependencies between parallel steps

In [ ]:
import time

rca_orchestrator_ex2 = Orchestrator(name="ParallelRCA")
rca_orchestrator_ex2.register_agent("log_parser", log_parser_agent)
rca_orchestrator_ex2.register_agent("pattern_detector", pattern_detector)
rca_orchestrator_ex2.register_agent("metric_analyzer", metric_analyzer)
rca_orchestrator_ex2.register_agent("correlator", correlator)
rca_orchestrator_ex2.register_agent("hypothesis_generator", hypothesis_generator)

workflow_ex2_sequential = [
    WorkflowStep(name="parse_logs", agent_name="log_parser"),
    WorkflowStep(name="detect_patterns", agent_name="pattern_detector", depends_on=["parse_logs"]),
    WorkflowStep(name="analyze_metrics", agent_name="metric_analyzer", depends_on=["detect_patterns"]),
    WorkflowStep(name="correlate_events", agent_name="correlator", depends_on=["analyze_metrics"]),
    WorkflowStep(name="generate_hypothesis", agent_name="hypothesis_generator", depends_on=["correlate_events"])
]

workflow_ex2_parallel = [
    WorkflowStep(name="parse_logs", agent_name="log_parser"),
    WorkflowStep(name="detect_patterns", agent_name="pattern_detector", depends_on=["parse_logs"]),
    WorkflowStep(name="analyze_metrics", agent_name="metric_analyzer", depends_on=["parse_logs"]),
    WorkflowStep(name="correlate_events", agent_name="correlator", depends_on=["detect_patterns", "analyze_metrics"]),
    WorkflowStep(name="generate_hypothesis", agent_name="hypothesis_generator", depends_on=["correlate_events"])
]

print("Sequential Workflow:")
rca_orchestrator_ex2.build_workflow(workflow_ex2_sequential)
start = time.time()
result_seq = rca_orchestrator_ex2.execute_workflow(rca_input_ex1, verbose=False)
seq_time = time.time() - start

print(f"\nSequential execution time: {seq_time:.2f}s")

print("\n" + "="*60)
print("Parallel Workflow:")
rca_orchestrator_ex2.reset()
rca_orchestrator_ex2.build_workflow(workflow_ex2_parallel)
start = time.time()
result_par = rca_orchestrator_ex2.execute_workflow(rca_input_ex1, verbose=False)
par_time = time.time() - start

print(f"\nParallel execution time: {par_time:.2f}s")
print(f"Time saved: {seq_time - par_time:.2f}s ({(1 - par_time/seq_time)*100:.1f}% faster)")

print("\nNote: In this MockLLM implementation, parallel execution doesn't show")
print("real time savings. With real LLMs and I/O operations, parallel execution")
print("can provide significant performance improvements.")

### Exercise 3: Multi-Incident Batch RCA

**Task**: Process multiple incidents and identify common root causes

**Requirements**:
- Run RCA on 5 incidents
- Identify patterns across incidents
- Generate summary report of common issues

**Hint**: Track hypotheses and look for duplicates

In [ ]:
from collections import Counter

print("\n" + "="*60)
print("Multi-Incident Batch RCA Analysis")
print("="*60 + "\n")

batch_rca_results = []
all_hypotheses = []

for i, incident in enumerate(incidents[:5], 1):
    print(f"\n{i}. Processing: {incident['id']} - {incident['title'][:50]}...")
    
    rca_orchestrator.reset()
    
    rca_input = {
        'incident': incident,
        'logs': sample_logs,
        'log_analysis': analysis
    }
    
    result = rca_orchestrator.execute_workflow(rca_input, verbose=False)
    
    hyp_result = rca_orchestrator.get_step_result("generate_hypothesis")
    
    batch_rca_results.append({
        'incident_id': incident['id'],
        'title': incident['title'],
        'severity': incident['severity'],
        'hypothesis': hyp_result['response'] if hyp_result else 'N/A',
        'status': result['status']
    })
    
    if hyp_result:
        all_hypotheses.append(hyp_result['response'])
    
    print(f"   Status: {result['status']}")

print("\n" + "="*60)
print("BATCH RCA SUMMARY REPORT")
print("="*60)

print(f"\nTotal Incidents Analyzed: {len(batch_rca_results)}")
print(f"Successful: {sum(1 for r in batch_rca_results if r['status'] == 'COMPLETED')}")

severity_counts = Counter(r['severity'] for r in batch_rca_results)
print(f"\nSeverity Distribution:")
for severity, count in severity_counts.most_common():
    print(f"  {severity}: {count}")

print(f"\nCommon Patterns Detected:")
common_terms = ['database', 'connection', 'memory', 'timeout', 'pool']
for term in common_terms:
    count = sum(1 for h in all_hypotheses if term.lower() in h.lower())
    if count > 0:
        print(f"  - '{term}' mentioned in {count} RCA reports")

print("\n" + "="*60)
print("Individual Results:")
print("="*60)
for result in batch_rca_results:
    print(f"\n{result['incident_id']}: {result['title'][:50]}...")
    print(f"  Severity: {result['severity']}")
    print(f"  Hypothesis: {result['hypothesis'][:100]}...")

---

## Part 6: Production Deployment

### 6.1 Deployment Checklist

In [ ]:
deployment_checklist = {
    'Infrastructure': [
        '☐ API keys stored in secure vault (not in code)',
        '☐ Rate limiting configured',
        '☐ Circuit breakers in place',
        '☐ Monitoring and alerting set up',
        '☐ Logging configured (audit trail)',
        '☐ Backup and recovery procedures'
    ],
    'Testing': [
        '☐ Unit tests for all agents',
        '☐ Integration tests for workflows',
        '☐ Load testing completed',
        '☐ Failure scenario testing',
        '☐ MockLLM testing passed',
        '☐ Real LLM testing passed'
    ],
    'Safety': [
        '☐ Human approval for critical actions',
        '☐ Confidence thresholds configured',
        '☐ Evidence validation enabled',
        '☐ Rollback procedures documented',
        '☐ Incident response plan ready',
        '☐ Guardrails tested and verified'
    ],
    'Documentation': [
        '☐ Agent responsibilities documented',
        '☐ Workflow diagrams created',
        '☐ Runbooks written',
        '☐ API documentation complete',
        '☐ Training materials prepared',
        '☐ Troubleshooting guide available'
    ]
}

print("PRODUCTION DEPLOYMENT CHECKLIST")
print("="*60)
for category, items in deployment_checklist.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  {item}")

### 6.2 Performance Metrics

In [ ]:
print_llm_stats(llm)

print("\nWorkflow Performance:")
summary = rca_orchestrator.get_workflow_summary()
print(json.dumps(summary, indent=2))

---

## Part 7: Key Takeaways

### What We Learned Today:

1. ✅ **Hierarchical systems** enable complex coordination
2. ✅ **Evidence-based reasoning** improves accuracy and trust
3. ✅ **5-agent RCA system** provides comprehensive analysis
4. ✅ **Confidence scoring** enables smart human-in-the-loop
5. ✅ **Guardrails** are essential for production safety
6. ✅ **Audit trails** provide accountability and debugging
7. ✅ **Parallel execution** improves performance

### Production Best Practices:

1. **Always validate evidence chains**
2. **Require human approval for critical actions**
3. **Log all decisions with reasoning**
4. **Implement rate limiting and circuit breakers**
5. **Monitor confidence scores**
6. **Test with MockLLM before deploying**
7. **Maintain comprehensive audit trails**
8. **Document agent responsibilities clearly**

### Architecture Patterns:

- **Sequential**: Simple, predictable, easier to debug
- **Parallel**: Faster, more complex, requires coordination
- **Hierarchical**: Scalable, modular, flexible
- **Hybrid**: Combines benefits of all patterns

### When to Use Multi-Agent Systems:

✅ **Good fit**:
- Complex tasks requiring multiple expertise areas
- Need for specialized analysis at each step
- Parallel processing opportunities
- Clear separation of concerns
- Need for audit trails and explainability

❌ **Not ideal**:
- Simple, single-step tasks
- Tight latency requirements
- Limited computational resources
- Tasks requiring deep context sharing

---

## 🎉 Course Complete!

Congratulations! You've built:
- ✅ 3-agent incident triage system (Day 1)
- ✅ 5-agent root cause analysis system (Day 2)
- ✅ Production guardrails and safety mechanisms
- ✅ Evidence-based reasoning pipelines
- ✅ Comprehensive monitoring and audit trails

### Next Steps:

1. **Apply to your use cases**: Adapt these patterns to your specific needs
2. **Experiment with real LLMs**: Set up `.env` and try with OpenAI
3. **Extend the systems**: Add more specialized agents
4. **Deploy to production**: Follow the deployment checklist
5. **Share your learnings**: Help others in your organization

### Resources:

- Code repository: All source code in `src/`
- Sample data: `data/` directory
- Documentation: `README.md`

### Feedback:

We'd love to hear about:
- What worked well
- What could be improved
- Your production use cases
- Additional topics you'd like covered

---

## Thank you for participating! 🚀

Keep building amazing multi-agent systems!